# Python Terraform Bootstrap

> nbdev source for Python-driven Terraform stack bootstrap helpers.


In [ ]:
#| default_exp terraform_bootstrap


In [ ]:
#| export
from __future__ import annotations

from dataclasses import dataclass
import json
from pathlib import Path
import subprocess
from typing import Any


@dataclass(frozen=True)
class TerraformStackConfig:
    """Minimal Terraform stack config for ML platform bootstrap."""

    stack_name: str = "ml-deploy-platform"
    aws_region: str = "us-east-1"
    mlflow_bucket_name: str = "ml-deploy-mlflow-artifacts"
    postgres_instance_class: str = "db.t4g.micro"
    postgres_allocated_storage: int = 20


def build_tfvars(config: TerraformStackConfig) -> dict[str, Any]:
    """Build Terraform variables as a dictionary."""
    return {
        "stack_name": config.stack_name,
        "aws_region": config.aws_region,
        "mlflow_bucket_name": config.mlflow_bucket_name,
        "postgres_instance_class": config.postgres_instance_class,
        "postgres_allocated_storage": config.postgres_allocated_storage,
    }


def write_stack_files(base_dir: Path, config: TerraformStackConfig) -> dict[str, Path]:
    """Generate Terraform JSON files from Python to avoid hand-written HCL."""
    base_dir.mkdir(parents=True, exist_ok=True)

    main_tf_json = {
        "terraform": {
            "required_providers": {"aws": {"source": "hashicorp/aws", "version": "~> 5.0"}}
        },
        "provider": {"aws": {"region": "${var.aws_region}"}},
        "variable": {
            "stack_name": {"type": "string"},
            "aws_region": {"type": "string"},
            "mlflow_bucket_name": {"type": "string"},
            "postgres_instance_class": {"type": "string"},
            "postgres_allocated_storage": {"type": "number"},
        },
        "resource": {
            "aws_s3_bucket": {
                "mlflow_artifacts": {"bucket": "${var.mlflow_bucket_name}"}
            },
            "aws_db_instance": {
                "mlflow_tracking": {
                    "identifier": "${var.stack_name}-mlflow",
                    "engine": "postgres",
                    "instance_class": "${var.postgres_instance_class}",
                    "allocated_storage": "${var.postgres_allocated_storage}",
                    "username": "mlflow",
                    "password": "mlflow-change-me",
                    "skip_final_snapshot": True,
                }
            },
        },
    }
    tfvars_json = build_tfvars(config)

    main_path = base_dir / "main.tf.json"
    vars_path = base_dir / "terraform.tfvars.json"
    main_path.write_text(json.dumps(main_tf_json, indent=2) + "\n", encoding="utf-8")
    vars_path.write_text(json.dumps(tfvars_json, indent=2) + "\n", encoding="utf-8")
    return {"main_tf_json": main_path, "terraform_tfvars_json": vars_path}


class PythonTerraformRunner:
    """Thin subprocess-based Terraform runner controlled from Python."""

    def __init__(self, working_dir: Path):
        self.working_dir = working_dir

    def _run(self, *args: str) -> subprocess.CompletedProcess[str]:
        return subprocess.run(
            ["terraform", *args],
            cwd=str(self.working_dir),
            text=True,
            capture_output=True,
            check=False,
        )

    def init(self) -> subprocess.CompletedProcess[str]:
        return self._run("init")

    def plan(self) -> subprocess.CompletedProcess[str]:
        return self._run("plan")

    def apply(self, *, auto_approve: bool = False) -> subprocess.CompletedProcess[str]:
        return self._run("apply", *(["-auto-approve"] if auto_approve else []))
